# tomato-agent: train on Colab and push the models to Git
**Before you run:** Runtime > Change runtime type > **T4 GPU**.
Add a Colab secret named `GITHUB_TOKEN` (key icon in the left bar, turn on *Notebook access*): a GitHub fine-grained token with *Contents: read and write* on your repo. It is read only inside the push cell and is never printed.

Then **Runtime > Run all**. The notebook:
1. clones your code from GitHub,
2. downloads the tomato images from the public PlantVillage and PlantDoc repos (images never go to your repo),
3. runs all 8 stages with `run_all.sh` (training is resumable; runs are kept on Google Drive),
4. pushes models, logits, metrics, thresholds and tables (**no images**) to the `colab-results` branch.

In [ ]:
#@title Settings
REPO_URL    = 'https://github.com/shekhar-ai99/tomato-agent.git'
CODE_BRANCH = 'main'            # branch with your code
PUSH_BRANCH = 'colab-results'   # branch that receives the trained models
BACKBONES   = 'efficientnet_b0 resnet50 vit_small_patch16_224'
SEEDS       = '0 1 2 3 4'       # pilot: BACKBONES='efficientnet_b0', SEEDS='0'
MAX_FILE_MB = 95                # GitHub rejects files over 100 MB; bigger ones are skipped (use Git LFS for them)

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))

## 1. Code

In [ ]:
import os, subprocess
%cd /content
!rm -rf tomato-agent
!git clone --branch $CODE_BRANCH $REPO_URL tomato-agent
%cd /content/tomato-agent
!pip install -q timm imagehash onnx onnxruntime scipy pandas matplotlib pytest
!python -m pytest -q

## 2. Keep training runs on Google Drive (survives a disconnect; re-run the notebook to resume)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/tomato_runs
    !rm -rf runs && ln -s /content/drive/MyDrive/tomato_runs runs
    print('runs/ -> Google Drive')
except Exception as e:
    print('Drive not mounted, runs stay on this VM only:', e)

## 3. Download the images (tomato only, ~1.1 GB, straight from the public repos)

In [ ]:
!bash scripts/download_data.sh

## 4. All 8 stages
About 5-8 h for 3 backbones x 5 seeds on a T4. If the session dies, open the notebook again and Run all: finished runs (with `metrics.json`) are skipped.

In [ ]:
os.environ['BACKBONES'] = BACKBONES
os.environ['SEEDS'] = SEEDS
os.environ['TRAIN_EXTRA'] = '--amp --workers 2'
!bash run_all.sh

## 5. Push to Git (no images)
Pushes to `PUSH_BRANCH`, never to `main`.

In [ ]:
from google.colab import userdata
import subprocess, pathlib

token = userdata.get('GITHUB_TOKEN')
sh = lambda *a, **k: subprocess.run(a, check=True, **k)

sh('git', 'config', 'user.name', 'colab-trainer')
sh('git', 'config', 'user.email', 'colab-trainer@users.noreply.github.com')
sh('git', 'checkout', '-B', PUSH_BRANCH)

IMG = [':(exclude)*.png', ':(exclude)*.jpg', ':(exclude)*.jpeg', ':(exclude)*.JPG', ':(exclude)*.gif', ':(exclude)*.bmp', ':(exclude)*.webp']
# .gitignore hides these folders, so add them with -f (images are excluded by the pathspecs)
for d in ['runs', 'manifests', 'artifacts', 'results', 'export', 'dataset_versions.txt']:
    if pathlib.Path(d).exists():
        sh('git', 'add', '-f', '--', d, *IMG)

# drop anything too big for GitHub
staged = subprocess.run(['git', 'diff', '--cached', '--name-only'], capture_output=True, text=True, check=True).stdout.splitlines()
for f in filter(None, staged):
    mb = pathlib.Path(f).stat().st_size / 1e6
    if mb > MAX_FILE_MB:
        sh('git', 'rm', '--cached', '-q', '--', f)
        print(f'skipped {f} ({mb:.0f} MB > {MAX_FILE_MB} MB)')

imgs = subprocess.run(['git', 'diff', '--cached', '--name-only'], capture_output=True, text=True, check=True).stdout
assert not any(x.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.webp')) for x in imgs.split()), 'an image got staged'
print(len(imgs.split()), 'files staged')

sh('git', 'commit', '-m', f'Colab results: {BACKBONES} seeds {SEEDS}')
url = REPO_URL.replace('https://', f'https://x-access-token:{token}@')
r = subprocess.run(['git', 'push', '--force', url, f'{PUSH_BRANCH}:{PUSH_BRANCH}'], capture_output=True, text=True)
print((r.stdout + r.stderr).replace(token, '***'))
r.check_returncode()
print('pushed to', REPO_URL.removesuffix('.git') + '/tree/' + PUSH_BRANCH)